# journey

> A goal and a path to it: an ordered list of steps, one concept each, that the learner can see and change.

A learner often wants to learn more than one concept. "I want to learn how LLMs work" needs tokenizers, embeddings, attention,
transformers and more, in a given order. A journey keeps that goal and that order on the learning shelf.

The planner (one model call) proposes the steps. It uses the concepts on the shelf when they fit. It makes new concepts for
the topics that the shelf does not have. Then the learner can change the path: move, skip, mark as known, add and split steps.

The journey does not keep progress. Progress comes from the learner model each time. A step is learned when the floor of its concept
reaches the goal depth. So a concept that the learner learns anywhere counts in each journey that has it.

In [ ]:
#| default_exp journey

In [ ]:
#| hide
from fastcore.test import *

Import the modules that this notebook uses.

In [ ]:
#| export
import json, math, random, time, uuid
from fastcore.utils import *
from abhyasa.store import Learning, slugify, _src
from abhyasa.model import GOAL, learned
from abhyasa.llm import ask_json
from abhyasa.sources import extract, _index, doc_pages, from_vault, placement
import abhyasa.profile  # adds Learning.profile
from abhyasa.method import check_for, EXAMPLES_TO, WALK_WAYS
from abhyasa.ground import ground, ground_text

## The document

A journey is one vault document of kind `journey`. Its meta has the goal, a title, a subject, and the steps. Each step has a slug,
a status (`todo`, `known` or `skip`) and the reason that it is in the path.

In [ ]:
#| export
STATUS = ('todo', 'known', 'skip')

def _jmeta(r): return AttrDict({**json.loads(r['meta'] or '{}'), 'doc_id': r['id']})

@patch
def put_journey(self:Learning, j:dict) -> AttrDict:
    "Save journey `j`, and replace an earlier one with the same id."
    j = {k: v for k, v in dict(j).items() if k != 'doc_id'}
    j.update(updated=time.time(), steps=[dict(s) for s in j.get('steps', [])])
    names = [(self.concept(s['slug']) or {}).get('name', s['slug']) for s in j['steps']]
    text = f"Journey: {j['title']} ({j['id']})\n\nGoal: {j.get('goal', '')}\n\nSteps: {', '.join(names)}"
    r = self.v.add(text, title=f"Journey: {j['title']}", source=_src('journey', j['id']), kind='journey', meta=j)
    if not r.get('changed'): self.v.set_meta(r['doc_id'], **j)
    return self.journey(j['id'])

@patch
def journey(self:Learning, jid:str) -> AttrDict|None:
    "One journey by id."
    r = first(self.v.t.docs(where='source=?', where_args=[_src('journey', jid)]))
    return _jmeta(r) if r else None

@patch
def journeys(self:Learning) -> L:
    "All journeys, the most recently changed first."
    return L(sorted(map(_jmeta, self._docs('journey')), key=lambda j: -j.get('updated', 0)))

@patch
def drop_journey(self:Learning, jid:str):
    "Remove a journey. Its concepts and their state stay."
    if (j := self.journey(jid)): self.v.forget(j.doc_id)

## Progress

`step_state` names the state of one step from its status and its concept. The states `learned`, `known`, `skipped` and
`missing` (the concept is gone) are done. The states `new` and `started` are not done.

In [ ]:
#| export
DONE = ('learned', 'known', 'skipped', 'missing')

def step_state(c, status:str='todo', goal:int=GOAL) -> str:
    "One of new, started, learned, known, skipped or missing."
    if status == 'known': return 'known'
    if status == 'skip': return 'skipped'
    if c is None: return 'missing'
    if learned(c, goal): return 'learned'
    return 'started' if c.get('seen') else 'new'

def progress(lr:Learning, j, goal:int=GOAL, by:dict=None) -> AttrDict:
    "Each step with its concept and state, the count done, the count that counts, and the current step."
    by = by if by is not None else {c.slug: c for c in lr.concepts()}
    steps = [AttrDict(s, c=by.get(s['slug']), name=(by.get(s['slug']) or {}).get('name', s['slug']),
                      state=step_state(by.get(s['slug']), s.get('status', 'todo'), goal)) for s in j['steps']]
    cur = first(s for s in steps if s.state not in DONE)
    return AttrDict(steps=steps, done=sum(s.state in ('learned', 'known') for s in steps),
                    total=sum(s.state not in ('skipped', 'missing') for s in steps), current=cur, finished=cur is None and bool(steps))

`current` gives the slug of the step to learn now. A step in `skip` waits, unless no other step is left.

In [ ]:
#| export
def current(lr:Learning, j, skip=(), goal:int=GOAL) -> str|None:
    "The slug of the first step that is not done and not in `skip`; else the first step that is not done."
    left = [s.slug for s in progress(lr, j, goal).steps if s.state not in DONE]
    return first(s for s in left if s not in skip) or first(left)

## Change the path

`journey_step` makes one change to one step. `journey_add` adds a step at the end, or before another step.

In [ ]:
#| export
@patch
def journey_step(self:Learning, jid:str, slug:str, act:str) -> AttrDict:
    "Do `act` on step `slug`: up, down, skip, known, todo or remove."
    j = self.journey(jid)
    if j is None: raise KeyError(f'no journey {jid!r}')
    ss = j.steps
    i = next((k for k, s in enumerate(ss) if s['slug'] == slug), None)
    if i is None: return j
    if act == 'up' and i > 0: ss[i - 1], ss[i] = ss[i], ss[i - 1]
    elif act == 'down' and i < len(ss) - 1: ss[i + 1], ss[i] = ss[i], ss[i + 1]
    elif act in STATUS: ss[i]['status'] = act
    elif act == 'remove': ss.pop(i)
    return self.put_journey(j)

@patch
def journey_add(self:Learning, jid:str, slug:str, before:str=None, why:str='') -> AttrDict:
    "Add concept `slug` as a step, at the end or before step `before`. A step that is there already stays where it is."
    j = self.journey(jid)
    if j is None: raise KeyError(f'no journey {jid!r}')
    if any(s['slug'] == slug for s in j.steps): return j
    at = next((k for k, s in enumerate(j.steps) if s['slug'] == before), len(j.steps))
    j.steps.insert(at, dict(slug=slug, status='todo', why=why))
    return self.put_journey(j)

## The planner

The planner gets the goal and the learner profile as text. It also gets the concepts on the shelf that are near the goal or that the
learner started. It returns the steps in order.

In [ ]:
#| export
PLAN_SP = """You plan a learning path toward one learner's goal.
- Give 4 to 9 steps, in the order to learn them. Each step is one idea that a learner can be asked about and can explain.
- Start where the learner is. Leave out what they know well: their background, and their concepts at depth 4 or more.
- When a known concept fits a step, give its slug in `slug`. Else `slug` is "".
- `statement` is true and self-contained: one to three sentences. Maths in $...$.
- `why` says in one short sentence why the step is in the path. Write to the learner as "you", in ASD-STE100 Simplified Technical English.
- `needs` names the earlier steps that this step needs first.
- `ways` names up to 3 ways that suit the idea best, from: code, maths, analogy, example, diagram, words, demo, order."""

def plan_prompt(goal:str, who:str='', known=()) -> str:
    "The prompt that plans a path to `goal`."
    kn = '\n'.join(f"- {s}: {n} (depth {f})" for s, n, f in known)
    return (f"Goal: {goal}\n" + (f"About the learner:\n{who}\n" if who else '')
            + (f"Concepts the learner has (slug: name (depth known)):\n{kn}\n" if kn else '')
            + '\nReturn JSON: {"title": str (3 to 6 words), "subject": str (one lower-case word, such as ml), '
            '"steps": [{"name": str, "statement": str, "slug": str, "why": str, "needs": [str], "ways": [str]}]}')

def near_goal(lr:Learning, goal:str, k:int=40, most:int=150) -> list:
    "`(slug, name, floor)` for the concepts that search finds for `goal`, and the ones the learner started."
    ids = {h['doc_id'] for h in lr.search(goal, kind='concept', limit=k)}
    return [(c.slug, c.name, c.floor) for c in lr.concepts() if c.doc_id in ids or c.get('seen')][:most]

`plan` saves the journey. A step can name a known slug, or a known name or alias. Then it uses that concept and keeps its state.
Each other step becomes a new concept, with the steps it needs as its prereqs. With `sources`, `plan` links each new concept to the
passages that name it in the learner's own sources.

In [ ]:
#| export
def _new_id(title): return f"{slugify(title)[:40]}-{uuid.uuid4().hex[:4]}"

def plan(lr:Learning, ask, goal:str, who:str='', sources=None) -> AttrDict:
    "Plan a journey to `goal`, make the concepts that the shelf does not have, and save it."
    r = ask_json(ask, plan_prompt(goal, who, near_goal(lr, goal)), PLAN_SP)
    idx, subject, steps, slug_of = _index(lr), (r.get('subject') or '').lower().strip(), [], {}
    for s in r.get('steps') or []:
        slug = s.get('slug') if s.get('slug') and lr.concept(s['slug']) else idx.get(s['name'].lower())
        if not slug:
            slug = slugify(s['name'])
            needs = [slug_of[n.lower()] for n in s.get('needs') or () if n.lower() in slug_of]
            lr.put(s['name'], s.get('statement', ''), subject, slug=slug, prereqs=needs, ways=[w for w in s.get('ways') or () if w in WALK_WAYS], provenance=[dict(src='journey', goal=goal)])
            idx[s['name'].lower()] = slug
            if sources:
                try: lr.connect(slug, sources)
                except Exception: pass
        slug_of[s['name'].lower()] = slug
        if slug not in [x['slug'] for x in steps]: steps.append(dict(slug=slug, status='todo', why=s.get('why', '')))
    title = r.get('title') or goal[:60]
    return lr.put_journey(dict(id=_new_id(title), title=title, goal=goal, subject=subject, steps=steps, created=time.time()))

## Walks

A walk is a journey with `mode='walk'`. It is the reading path: one step for each idea, in reading order. Each step has the passage to
read (`read`) and the check to do (`check`). `progress` does not change.

`_check` picks the check of one concept. It uses the stats and the profile of the learner, as `Tutor.ask_on` does.

In [ ]:
#| export
def _check(lr:Learning, c) -> AttrDict:
    "The check for concept `c`, from `check_for` with the stats and the profile of the learner."
    p = lr.profile()
    return check_for(c, lr.stats(), random.Random(), p.get('explore', 'sometimes'), [EXAMPLES_TO[e] for e in p.examples if e in EXAMPLES_TO])

`journey_from_source` extracts the ideas of one document, and makes one step for each idea of that document, in the order of the document.
A step has the passage that `extract` kept. A concept that you know at depth 4 or more has the status `known`.

In [ ]:
#| export
def _walk(lr:Learning, jid:str, title:str, slugs, doc_id:str, source:dict|None, goal:str, subject:str) -> AttrDict:
    "Save a walk with one step for each of `slugs` that has a place in document `doc_id`, in the order of the document."
    cs = sorted(((c, placement(c, doc_id)) for c in map(lr.concept, slugs) if c and placement(c, doc_id)), key=lambda x: x[1].get('order') or 0)
    steps = [dict(slug=c.slug, status='known' if (c.get('floor') or 0) >= 4 else 'todo', why='',
                  read=dict(shelf=(source or {}).get('shelf', 'store'), doc_id=doc_id, title=title, section=p.get('section') or '',
                            page=p.get('page'), span=p.get('span'), text=p.get('passage') or '', figure=p.get('figure')),
                  check=_check(lr, c)) for c, p in cs]
    return lr.put_journey(dict(id=jid, title=title, goal=goal, subject=subject, steps=steps, created=time.time(),
                               mode='walk', source=source, started=None, sitting=None))

def journey_from_source(lr:Learning, ask, ref:str, shelf:str='store', subject:str='') -> AttrDict:
    "Make a walk from one vault document, by doc id, source or title."
    d = doc_pages(lr, ref, shelf)
    slugs = from_vault(lr, ask, d.doc_id, subject, shelf)
    return _walk(lr, _new_id(d.title), d.title, slugs, d.doc_id, dict(shelf=shelf, doc_id=d.doc_id, title=d.title), f'Read {d.title}', subject)

`journey_from_text` does the same for pasted text. A passage shows only when its quote is in the text word for word.

In [ ]:
#| export
def journey_from_text(lr:Learning, ask, text:str, title:str, subject:str='') -> AttrDict:
    "Make a walk from pasted `text`."
    jid = _new_id(title)
    slugs = extract(lr, ask, subject=subject, source=dict(src='text', doc_id=jid, title=title), pages=[(None, text)])
    return _walk(lr, jid, title, slugs, jid, None, f'Read {title}', subject)

`walk_from_topic` plans a path to the goal. Then each step reads from the first passage in your sources that names the idea.
A step has no `read` when no passage names it.

In [ ]:
#| export
def _topic_read(c, sources) -> dict|None:
    "The first passage in `sources` that names concept `c`, as a `read`, or None."
    name = c.get('name') or c.slug
    p = first(ground(name, sources, 1, must=[name, *(c.get('aliases') or ())])) if sources else None
    return p and dict(src=p.src, ref=p.ref, title=p.title, url=p.url, text=p.text[:600], section='', figure=None)

def walk_from_topic(lr:Learning, ask, goal:str, who:str='', sources=None) -> AttrDict:
    "Plan a path to `goal` and make it a walk, with a passage from `sources` and a check on each step."
    j = plan(lr, ask, goal, who, sources)
    for s in j.steps:
        c = lr.concept(s['slug'])
        s.update(read=_topic_read(c, sources), check=_check(lr, c))
    return lr.put_journey({**j, 'mode': 'walk', 'source': None, 'started': None, 'sitting': None})

## Change a step, read the rail

`walk_step` merges changes into one step and saves the walk. `walk_state` gives each step a dot for the rail: `done` (outcome `solid`),
`shaky` (outcome `fragile`, `gap` or `misconception`), `skipped`, `known`, `now` (the first step with no outcome and the status `todo`)
or `todo`.

In [ ]:
#| export
@patch
def walk_step(self:Learning, jid:str, i:int, **kv) -> AttrDict:
    "Merge `kv` into step `i` of walk `jid` and save it."
    j = self.journey(jid)
    if j is None: raise KeyError(f'no journey {jid!r}')
    if not -len(j.steps) <= i < len(j.steps): raise IndexError(f'no step {i} in journey {jid!r}')
    j.steps[i].update(kv)
    return self.put_journey(j)

SHAKY = ('fragile', 'gap', 'misconception')

def _dot(s, now:bool) -> str:
    o = (s.get('done') or {}).get('outcome')
    if o == 'solid': return 'done'
    if o in SHAKY: return 'shaky'
    if o == 'skipped' or s.get('status') == 'skip': return 'skipped'
    if s.get('status') == 'known': return 'known'
    return 'now' if now else 'todo'

def walk_state(j) -> AttrDict:
    "The steps with a `dot`, the index of the current step, the counts, and the sections as `(name, [indexes])`."
    cur = next((i for i, s in enumerate(j['steps']) if not s.get('done') and s.get('status', 'todo') == 'todo'), None)
    steps = [AttrDict(s, dot=_dot(s, i == cur)) for i, s in enumerate(j['steps'])]
    secs = []
    for i, s in enumerate(steps):
        name = (s.get('read') or {}).get('section') or ''
        if secs and secs[-1][0] == name: secs[-1][1].append(i)
        else: secs.append((name, [i]))
    return AttrDict(steps=steps, current=cur, done=sum(s.dot in ('done', 'known') for s in steps),
                    total=sum(s.dot != 'skipped' for s in steps), sections=secs)

`minutes` guesses the time for one step: reading at 200 words a minute, plus the time of its check.

In [ ]:
#| export
STEP_MIN = dict(short=2, order=2, demo=3, code=6)

def minutes(step) -> int:
    "The minutes for one step: its passage at 200 words a minute, plus the time of its check."
    words = len(((step.get('read') or {}).get('text') or '').split())
    return math.ceil(words / 200) + STEP_MIN.get((step.get('check') or {}).get('kind'), STEP_MIN['short'])

## Add a topic, split a step

`add_topic` adds a topic by name. When the shelf knows the name, `add_topic` uses that concept. For a new name, the model writes a short statement.

In [ ]:
#| export
PRIMER_SP = "You write one true, self-contained statement of one idea, for a learner: one to three sentences. Maths in $...$."

def add_topic(lr:Learning, ask, jid:str, name:str, before:str=None, sources=None) -> AttrDict:
    "Add topic `name` to journey `jid`, as a concept on the shelf. Return the journey."
    j = lr.journey(jid)
    slug = _index(lr).get(name.strip().lower())
    if not slug:
        r = ask_json(ask, f"The idea: {name}\nThe path it is part of: {j.title} ({j.get('goal', '')})\n\n"
                          'Return JSON: {"name": str, "statement": str}', PRIMER_SP)
        slug = lr.put(r.get('name') or name, r.get('statement', ''), j.get('subject', ''), provenance=[dict(src='journey', journey=jid)]).slug
        if sources:
            try: lr.connect(slug, sources)
            except Exception: pass
    return lr.journey_add(jid, slug, before, why='you added it')

`name_gaps` keeps each idea that you say you do not know as a gap of the concept you learn. It finds the idea on the shelf by name or alias. Or it adds the idea with a short statement. Next time the gap comes first, but it does not block the concept.

In [ ]:
#| export
def name_gaps(lr:Learning, ask, slug:str, terms, sources=None) -> L:
    "Find or add a concept for each idea in `terms`, and keep it as a gap of `slug`. Return the gap slugs."
    c, idx, out = lr.concept(slug), _index(lr), L()
    for t in dict.fromkeys(str(t).strip() for t in terms or () if str(t).strip()):
        if not (g := idx.get(t.lower())):
            r = ask_json(ask, f"The idea: {t}\nIt comes up in: {c.name} ({c.get('subject') or 'general'})\n\n"
                              'Return JSON: {"name": str, "statement": str}', PRIMER_SP)
            name = r.get('name') or t
            g = lr.put(name, r.get('statement', ''), c.get('subject', ''), aliases=[t] if t.lower() != name.lower() else [],
                       provenance=[dict(src='gap', of=slug, said=t, at=time.time())]).slug
            if sources:
                try: lr.connect(g, sources)
                except Exception: pass
        lr.add_gap(slug, g); out.append(g)
    return out

A new idea becomes a concept with a statement and the gap provenance. `name_gaps` finds an idea that is already on the shelf by its alias. It does not add that idea again.

In [ ]:
from abhyasa.store import temp_learning
from abhyasa.llm import Fake
ngl = temp_learning(); ngl.put('Functions and the neuron', 'z = w . x + b.', 'ml'); ngl.put('Bias', 'A constant added to z.', 'ml', aliases=['the bias'])
gask = Fake({'The idea: the loop': dict(name='Weighted sum as a loop', statement='A loop adds each weight times its input.')})
test_eq(name_gaps(ngl, gask, 'functions-and-the-neuron', ['the loop', 'the bias', 'the loop']), ['weighted-sum-as-a-loop', 'bias'])
g = ngl.concept('weighted-sum-as-a-loop')
test_eq((g.aliases, g.provenance[0]['src'], g.provenance[0]['of']), (['the loop'], 'gap', 'functions-and-the-neuron'))
test_eq(ngl.concept('functions-and-the-neuron').gaps, ['weighted-sum-as-a-loop', 'bias'])

`split` divides a big step into smaller steps. It runs `extract` over the statement of the step and the passages from the
learner's sources. The concepts that it finds go before the step, as its prereqs.

In [ ]:
#| export
def split(lr:Learning, ask, jid:str, slug:str, sources=None) -> L:
    "Find the parts of step `slug`, add them before it as steps and prereqs, and return their slugs."
    c, j = lr.concept(slug), lr.journey(jid)
    ps = ground(' '.join([c.name, *(c.aliases or [])][:4]), sources, 4) if sources else L()
    text = f"{c.name}\n\n{c.statement}" + (f"\n\nPassages:\n{ground_text(ps)}" if ps else '')
    have = {s['slug'] for s in j.steps}
    new = L(s for s in extract(lr, ask, text, c.subject, dict(src='journey', journey=jid, split=slug)) if s != slug and s not in have)
    for s in new:
        lr.link(slug, s)
        lr.journey_add(jid, s, before=slug, why=f'a part of {c.name}')
    return new

## Related topics

`related` gives the concepts next to `slug` in the graph. First come the concepts that build on it, then the related ones, then its
prereqs. The concepts that are not learned come first. `with_step` finds the journeys that have a concept. `beyond` gives what comes
after a journey: the concepts that build on its steps and are not in it.

In [ ]:
#| export
def related(lr:Learning, slug:str, k:int=6, goal:int=GOAL, by:dict=None) -> L:
    "Up to `k` concepts next to `slug`: what builds on it, what relates to it, then what it needs."
    by = by if by is not None else {c.slug: c for c in lr.concepts()}
    c = by.get(slug)
    if c is None: return L()
    up = [x for x in by.values() if slug in (x.get('prereqs') or ()) or slug in (x.get('related') or ())]
    near = [*up, *(by[s] for s in c.get('related') or () if s in by), *(by[s] for s in c.get('prereqs') or () if s in by)]
    seen, out = {slug}, []
    for x in sorted(near, key=lambda x: learned(x, goal)):   # a stable sort keeps the order of kinds
        if x.slug not in seen: seen.add(x.slug); out.append(x)
    return L(out[:k])

def with_step(lr:Learning, slug:str) -> L:
    "`(journey, index)` for each journey that has step `slug`."
    return L((j, i) for j in lr.journeys() for i, s in enumerate(j.steps) if s['slug'] == slug)

def beyond(lr:Learning, j, k:int=5, goal:int=GOAL) -> L:
    "Up to `k` concepts that build on the steps of journey `j` and are not in it."
    by, have = {c.slug: c for c in lr.concepts()}, {s['slug'] for s in j['steps']}
    out = L()
    for s in j['steps'][::-1]:
        out += [x for x in related(lr, s['slug'], k, goal, by) if x.slug not in have and x.slug not in out.attrgot('slug')]
    return out[:k]

## Tests

A fake model plans a path to LLMs. The shelf has `attention` already, at depth 3.

In [ ]:
from abhyasa.store import temp_learning
from abhyasa.llm import Fake
lr = temp_learning()
lr.put('Attention', 'Weights from query-key dot products.', 'ml')
lr.update('attention', floor=3, seen=1)
fake = Fake({'You plan a learning path': dict(title='How LLMs work', subject='ml', steps=[
    dict(name='Tokenizers', statement='A tokenizer cuts text into tokens.', slug='', why='The model reads tokens.', needs=[]),
    dict(name='Embeddings', statement='An embedding maps a token to a vector.', slug='', why='Tokens become vectors.', needs=['Tokenizers']),
    dict(name='Attention', statement='', slug='attention', why='The core of a transformer.', needs=['Embeddings']),
    dict(name='Mamba', statement='Mamba is a selective state-space model.', slug='', why='Another way to mix tokens.', needs=['Attention'])])})
j = plan(lr, fake, 'I want to learn how LLMs work', who='Background: codes in Python')

`plan` keeps the order, uses the known concept, and makes the new ones with their prereqs.

In [ ]:
test_eq([s['slug'] for s in j.steps], ['tokenizers', 'embeddings', 'attention', 'mamba'])
test_eq((lr.concept('embeddings').prereqs, lr.concept('mamba').prereqs, lr.concept('mamba').subject), (['tokenizers'], ['attention'], 'ml'))
test_eq(lr.concept('attention').floor, 3)    # a known concept keeps its state
assert 'codes in Python' in fake.calls[-1][0] and 'attention: Attention (depth 3)' in fake.calls[-1][0]

`progress` names the state of each step. The current step is the first that is not done.

In [ ]:
pg = progress(lr, j)
test_eq([s.state for s in pg.steps], ['new', 'new', 'started', 'new'])
test_eq((pg.done, pg.total, pg.current.slug), (0, 4, 'tokenizers'))

A step that the learner learns anywhere is learned in this journey too. A step that is marked known or skipped is done. Then `current` moves to the next step.

In [ ]:
lr.update('tokenizers', floor=4, seen=2)
lr.journey_step(j.id, 'embeddings', 'known')
pg = progress(lr, lr.journey(j.id))
test_eq([s.state for s in pg.steps], ['learned', 'known', 'started', 'new'])
test_eq((pg.done, current(lr, lr.journey(j.id)), current(lr, lr.journey(j.id), skip={'attention'})), (2, 'attention', 'mamba'))

Steps move up and down. A removed step leaves the path. A skipped step does not count in the total.

In [ ]:
lr.journey_step(j.id, 'mamba', 'up')
test_eq([s['slug'] for s in lr.journey(j.id).steps], ['tokenizers', 'embeddings', 'mamba', 'attention'])
lr.journey_step(j.id, 'mamba', 'skip')
test_eq(progress(lr, lr.journey(j.id)).total, 3)
lr.journey_step(j.id, 'mamba', 'remove')
test_eq(len(lr.journey(j.id).steps), 3)

`add_topic` uses a known name, and asks the model only for a new one.

In [ ]:
fake.rules['The idea: '] = dict(name='PyTorch', statement='PyTorch is a Python library for tensors and autograd.')
add_topic(lr, fake, j.id, 'PyTorch')
add_topic(lr, fake, j.id, 'tokenizers')    # known, and in the path already: no change
test_eq([s['slug'] for s in lr.journey(j.id).steps], ['tokenizers', 'embeddings', 'attention', 'pytorch'])

`split` adds the parts of a step before it, as its prereqs.

In [ ]:
fake.rules['Return JSON: {"concepts"'] = dict(concepts=[dict(name='Query and key', statement='Each token has a query and a key.', aliases=[], prereqs=[])])
new = split(lr, fake, j.id, 'attention')
test_eq(list(new), ['query-and-key'])
test_eq([s['slug'] for s in lr.journey(j.id).steps], ['tokenizers', 'embeddings', 'query-and-key', 'attention', 'pytorch'])
assert 'query-and-key' in lr.concept('attention').prereqs

`related` puts what builds on a concept first. `with_step` finds the journeys of a concept. `beyond` gives what comes after.

In [ ]:
lr.put('Transformer block', 'Attention, then a feed-forward layer.', 'ml', prereqs=['attention'])
test_eq(related(lr, 'attention').attrgot('slug'), ['mamba', 'transformer-block', 'query-and-key'])   # mamba left the path, but it builds on attention
test_eq([(x.id, i) for x, i in with_step(lr, 'attention')], [(j.id, 3)])
test_eq(beyond(lr, lr.journey(j.id)).attrgot('slug'), ['mamba', 'transformer-block'])

`journeys` lists the journeys with the newest change first. `drop_journey` removes one journey.

In [ ]:
j2 = lr.put_journey(dict(id='sanskrit', title='Read the Gita', goal='read', steps=[]))
test_eq(lr.journeys().attrgot('id'), ['sanskrit', j.id])
lr.drop_journey('sanskrit')
test_eq(len(lr.journeys()), 1)

## Tests for walks

A walk is a journey with `mode='walk'`. These tests use the small paper in `tests/walk_data.py` and a fake model.

In [ ]:
#| hide
import sys; sys.path.insert(0, '../tests'); from walk_data import *
from abhyasa.ground import passage

A walk from a source has one step for each idea of the paper, in the order of the paper. Each step has the passage to read and a check.

In [ ]:
wl = temp_learning(); did = add_paper(wl)
ask = Fake({'Return JSON: {"concepts"': EXTRACT_REPLY})
wj = journey_from_source(wl, ask, did)
assert wj.mode == 'walk' and wj.source == dict(shelf='store', doc_id=did, title='Attention Is All You Need') and wj.started is None
assert [s['slug'] for s in wj.steps] == ['sequential-computation', 'softmax', 'scaling-by-the-root-of-dk']
r = wj.steps[1]['read']
assert (r['section'], r['page'], r['figure']) == ('3.2.1 Scaled Dot-Product Attention', 4, 'attention/images/fig2.png')
assert wj.steps[1]['check']['kind'] in ('code', 'short') and wj.steps[2]['check']['way'] in ('demo', 'maths')
assert progress(wl, wj).total == 3     # progress does not change

`walk_state` gives each step a dot for the rail. `walk_step` keeps a change on one step.

In [ ]:
ws = walk_state(wl.walk_step(wj.id, 0, done=dict(outcome='fragile', aid='a1', at=1.)))
assert [s.dot for s in ws.steps] == ['shaky', 'now', 'todo'] and ws.current == 1
assert ws.sections == [('1 Introduction', [0]), ('3.2.1 Scaled Dot-Product Attention', [1, 2])]
assert minutes(wj.steps[1]) >= 2
test_fail(lambda: wl.walk_step('none', 0), contains='none', exc=KeyError)
test_fail(lambda: wl.walk_step(wj.id, 9), exc=IndexError)

A concept that you know at depth 4 or more stays in the walk, as known.

In [ ]:
wl.update('softmax', floor=4)
assert journey_from_source(wl, ask, did).steps[1]['status'] == 'known'

A walk from pasted text takes the passage only when its quote is in the text. An idea with no quote in the text has no passage.

In [ ]:
jt = journey_from_text(wl, ask, PAGES[1][1], 'Attention notes')
sm = next(s for s in jt.steps if s['slug'] == 'softmax')
assert jt.source is None and sm['read']['text'].startswith('apply a softmax function')
assert next(s for s in jt.steps if s['slug'] == 'sequential-computation')['read']['text'] == ''

A walk from a topic plans the path, then reads from the first passage in your sources that names each idea. With no sources, a step has no passage.

In [ ]:
PLAN = dict(title='Attention', subject='ml', steps=[dict(name='Softmax', statement='', slug='softmax', why='Weights.', needs=[], ways=['code'])])
class Notes:
    name = 'notes'
    def search(self, q, k=4): return [passage('vault:notes', 'n1', 'My notes', 'Softmax turns scores into weights.')]
jw = walk_from_topic(wl, Fake({'You plan a learning path': PLAN}), 'learn attention', sources=[Notes()])
assert jw.mode == 'walk' and jw.steps[0]['read']['text'].startswith('Softmax turns') and jw.steps[0]['check']
assert walk_from_topic(wl, Fake({'You plan a learning path': PLAN}), 'learn attention', sources=[]).steps[0]['read'] is None

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()